# 月データ探索ツール（発展編）

`explore.ipynb`（標準編）で「気になる関係」を見つけた人向けに、回帰・べき乗則フィットなど
情報Ⅰの範囲を超える内容（`numpy.polyfit`・`seaborn`）まで踏み込むノートブックです。

対象：数学や地学基礎で相関・回帰に触れたことがある人、SSH／理数科の探究活動など。

構成：
- **A. べき乗則フィット**：クレーターの直径分布は「べき乗則」に従うことが知られています。
  両対数グラフで直線になるかどうかを確認し、傾き（べき指数）を回帰で求めます。

> 月齢と地球環境（地震・潮汐力）の相関を扱う教材は、教育的リスク（相関が無いことを示す
> 構成自体が「月と地震に関係がある」という誤解を招く懸念）を理由に、教材のスコープから
> 完全に除外しています（MVP確定版 Ver.1.4）。

In [ ]:
import sys, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB and not os.path.exists('data'):
    print("⚠️ dataフォルダが見つかりません。explore.ipynbの案内を参照してリポジトリごと取得してください。")

def _read(name):
    for p in (f'../data/{name}', f'data/{name}'):
        if os.path.exists(p):
            return pd.read_csv(p)
    raise FileNotFoundError(name)

craters = _read('craters_subset.csv')

# 日本語フォントの設定（explore.ipynbと同様、同梱のNoto Sans JPを使用）
# seabornのset_themeがrcParamsをリセットするため、set_theme実行後にもう一度設定する。
_font_path = None
for _p in ('assets/NotoSansJP-Regular.ttf', 'notebooks/assets/NotoSansJP-Regular.ttf'):
    if os.path.exists(_p):
        _font_path = _p
        break
if _font_path:
    fm.fontManager.addfont(_font_path)
    _jp_font_name = fm.FontProperties(fname=_font_path).get_name()
else:
    _jp_font_name = None
    print("⚠️ 日本語フォント(assets/NotoSansJP-Regular.ttf)が見つかりません。"
          "グラフの日本語表示が文字化けする可能性があります。")

sns.set_theme(style='whitegrid')
if _jp_font_name:
    plt.rcParams['font.family'] = _jp_font_name

print(f'craters: {len(craters):,}件')

## A. クレーター直径のべき乗則フィット

「直径Dより大きいクレーターの個数 N(>D)」を数え、横軸・縦軸をともに対数にして散布図を描くと、
多くの天体でおおむね直線（＝べき乗則 $N(>D) \propto D^{-b}$）になることが知られています。
`numpy.polyfit` で対数-対数の傾き（べき指数 $b$）を最小二乗法で求めてみましょう。

In [ ]:
diam = np.sort(craters['diam_km'].to_numpy())[::-1]
n_cumulative = np.arange(1, len(diam) + 1)  # diam[i]より大きい(以上の)クレーターの個数

log_d = np.log10(diam)
log_n = np.log10(n_cumulative)

slope, intercept = np.polyfit(log_d, log_n, 1)
fit_n = 10 ** (slope * log_d + intercept)

fig, ax = plt.subplots(figsize=(7, 6))
ax.loglog(diam, n_cumulative, '.', markersize=3, alpha=0.4, label='実データ')
ax.loglog(diam, fit_n, 'r-', linewidth=2, label=f'べき乗則フィット（傾き b = {slope:.2f}）')
ax.set_xlabel('直径 D [km]（対数）')
ax.set_ylabel('直径D以上のクレーター累積個数 N(>D)（対数）')
ax.set_title('クレーターサイズ頻度分布（累積）')
ax.legend()
plt.tight_layout()
plt.show()

print(f'べき指数 b \u2248 {abs(slope):.2f}（一般的な文献値はおよそ2前後）')

---
## B. クレーターの「数」から「絶対年代」を出す（SSH 向け）

標準の教材（ステップ2）では「海のほうがクレーターが少ない＝新しい」まで（**相対**年代）でした。
ここでは **単位面積あたりのクレーターの数** から **○○億年前** という数字（**絶対**年代）を出します。

原理：隕石が降る割合は数十億年ほぼ一定なので、古い地面ほどクレーターが溜まっている。
`load('アイソクロン')` は、Neukum の生産関数＋月の編年関数（Neukum et al. 2001）から計算した
**「その年代の地面なら、直径8km以上のクレーターが 100万km² あたり何個あるはず」** の参照表です。

**やること**：海の一区画を自分で選ぶ → その中の直径8km以上のクレーターを数える →
面積で割って密度を出す → 参照表と照らして年代を推定する。

In [ ]:
from moonkit import box_area_km2

iso = _read('isochron_reference.csv')
print(iso.to_string(index=False))

# ★ここを変える：調べたい海の一区画（雨の海のあたり）。高地が入らないように少し内側に取る
my_lat = (22, 42)
my_lon = (-35, -8)

box = craters[(craters.lat.between(*my_lat)) & (craters.lon.between(*my_lon)) & (craters.diam_km >= 8)]
area_Mkm2 = box_area_km2(my_lat, my_lon) / 1e6
density = len(box) / area_Mkm2
print()
print('区画：緯度', my_lat, '経度', my_lon)
print('  直径8km以上のクレーター', len(box), '個 / 面積', round(area_Mkm2, 3), '百万km2')
print('  密度 =', round(density, 1), '個 / 百万km2')

import numpy as np
age = np.interp(density, iso.N_ge_8km_per_Mkm2, iso.age_Ga)
print('  → 推定年代 ≈', round(float(age), 2), 'Ga（十億年前）')
print('  文献：雨の海の玄武岩は約 3.3〜3.6 Ga（Hiesinger ほか）')

In [ ]:
# 誤差を体感：区画の取り方・下限直径を変えると年代はどれくらい動く？
import numpy as np
print('区画を少しずつ変えてみる：')
for dlat, dlon in [(0, 0), (3, 0), (-3, 0), (0, 4), (0, -4), (5, 5), (-5, -5)]:
    la = (my_lat[0] + dlat, my_lat[1] + dlat)
    lo = (my_lon[0] + dlon, my_lon[1] + dlon)
    b = craters[(craters.lat.between(*la)) & (craters.lon.between(*lo)) & (craters.diam_km >= 8)]
    d = len(b) / (box_area_km2(la, lo) / 1e6)
    a = np.interp(d, iso.N_ge_8km_per_Mkm2, iso.age_Ga)
    print('  緯度', la, '経度', lo, ':', len(b), '個  密度', round(d, 1), '→', round(float(a), 2), 'Ga')

print()
print('下限直径を変えてみる（区画は固定。20km以上で数える）：')
for dmin, col in [(8, 'N_ge_8km_per_Mkm2'), (20, 'N_ge_20km_per_Mkm2')]:
    b = craters[(craters.lat.between(*my_lat)) & (craters.lon.between(*my_lon)) & (craters.diam_km >= dmin)]
    d = len(b) / (box_area_km2(my_lat, my_lon) / 1e6)
    a = np.interp(d, iso[col], iso.age_Ga)
    print('  直径', dmin, 'km以上:', len(b), '個  密度', round(d, 1), '→', round(float(a), 2), 'Ga')

print()
print('→ 年代は 0.1 Ga 前後しか動かない。海の年代（~3.5 Ga）では密度→年代の曲線が急なので、')
print('  「クレーター年代学で ○○ Ga」にはこういう幅が必ずある。')

**読み取り（ワークシート）**：
- あなたの区画の推定年代は？　文献の「雨の海 3.3〜3.6 Ga」と合っている？
- 区画を動かすと年代は何 Ga 動いた？（海の年代では 0.1 Ga 程度。密度→年代の曲線が急なため）
- 高地（例：緯度 -50〜-30、経度 -20〜10 あたり）で同じことをすると？　4 Ga を超える？
- **この方法の限界**：区画が小さいとクレーターの数が少なく統計が不安定／二次クレーター
  （大きな衝突の破片が作る小クレーター）が混ざると数えすぎる／参照表は特定のモデル
  （Neukum 2001）に依存する。別のモデル（Hartmann）だと少しずれる。
- **相対年代（ステップ２）との違い**：ステップ２では「海は陸より新しい」までだった。
  ここではモデルを一つ選んで「約 3.5 Ga」という数字まで踏み込んだ。数字にはモデル依存の幅がある。

## まとめ：回帰を読むときの注意点

1. **回帰直線（べき乗則フィット）はデータの傾向を要約するものであり、すべての点にぴったり
   当てはまるわけではない**。実データが直線からずれている範囲（今回は特に大きい直径側）にも
   注目してみましょう。
2. **対数グラフでは、見た目の距離が「比率」を表す**ことに注意してください。対数目盛りの
   グラフ上で近く見えても、元の値では大きく違うことがあります。
3. べき指数のような要約統計量は、生データの分布そのもの（ヒストグラムや散布図）と
   あわせて確認することが大切です。

## データの出典

- Robbins, S. J. (2018). *A New Global Database of Lunar Impact Craters >1–2 km*. USGS Astrogeology Science Center.
  https://astrogeology.usgs.gov/search/map/Moon/Research/Craters/lunar_crater_database_robbins_2018